<a href="https://colab.research.google.com/github/subu604/gen-ai/blob/main/simpleRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install the Google Generative AI SDK
!pip install -q openai google-generativeai sentence_transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 2.9 MB/s eta 0:00:00


In [3]:
!pip install -q openai

In [4]:
# Step 2: Import libraries
import os
import openai
from openai import OpenAI
from google.colab import userdata
#import google.generativeai as genai
from IPython.display import display, Markdown


# Step 3
GOOGLE_API_KEY=userdata.get('openai_key')
GEMINI_API_KEY=GOOGLE_API_KEY
gemini_model='gpt-4.1-mini-2025-04-14'
# gemini_api_endpoint="https://generativelanguage.googleapis.com/v1beta/openai/"
# gemini_api_endpoint='https://api.openai.com/v1/chat/completions'
gemini_api_endpoint = 'https://api.openai.com/v1'

In [5]:
# Step 4: Initialize the model (using Gemini flash)
# Configure OpenAI client to use Google's Gemini endpoint
gemini = OpenAI(api_key=GOOGLE_API_KEY, base_url=gemini_api_endpoint)

In [6]:

!echo "i am subu. i work at bla bla company. i am from Hyderbad India. I love to travel" > "sample_ai_textbook.txt"

In [7]:

# Step 1: Load and chunk document
with open("sample_ai_textbook.txt", "r") as file:
    text = file.read()


In [8]:
# Simple chunking by splitting into paragraphs (approx. 500 chars)
chunks = [text[i:i+500] for i in range(0, len(text), 500)]
print(f"Created {len(chunks)} chunks")

Created 1 chunks


In [9]:

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
#from huggingface_hub import InferenceClient

In [10]:
# Step 2: Create embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, convert_to_numpy=True)
dimension = embeddings.shape[1]  # Embedding size (e.g., 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# Step 3: Store embeddings in FAISS
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Embeddings stored in FAISS")

Embeddings stored in FAISS


In [12]:

def retrive_chunk(question):
  question_embedding = model.encode([question])[0]
  D, I = index.search(np.array([question_embedding]), k=1)  # Find top 1 chunk
  retrieved_chunk = chunks[I[0][0]]
  print("Retrieved chunk:", retrieved_chunk[:100], "...")
  return retrieved_chunk

In [13]:


def getAnswer(question) :
  retrieved_chunk = retrive_chunk(question)
  system="you are a LLM using RAG to answer user questions. you are impersonating the person mentioned in RAG. so strictly build answers based on context. If you donot know answer, say i dont know  "
  prompt = f"Context: {retrieved_chunk}\n\nQuestion: {question}\nAnswer:"
  answer = gemini.chat.completions.create(
      model=gemini_model,
      messages=[
          {"role": "system", "content": system},
          {"role": "user", "content": prompt}
      ]
  ).choices[0].message.content
  return answer


In [14]:

# Step 4: Query the system
question = "whati like todo?"
answer=getAnswer(question)
print("Question:", question)
print("Answer:", answer)


Retrieved chunk: i am subu. i work at bla bla company. i am from Hyderbad India. I love to travel
 ...
Question: whati like todo?
Answer: You like to travel.
